In [56]:
import pandas as pd
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [57]:
df = pd.read_csv("../data/1_Recipe_csv.csv")
df.head()

,recipe_title,category,subcategory,description,ingredients,directions,num_ingredients,num_steps
0,Air Fryer Potato Slices with Dipping Sauce,Air Fryer Recipes,Air Fryer Recipes,"These air fryer potato slices, served with a b...","[""3/4 cup ketchup"", ""1/2 cup beer"", ""1 tablesp...","[""Combine ketchup, beer, Worcestershire sauce,...",9,5
1,Gochujang Pork Belly Bites,Air Fryer Recipes,Air Fryer Recipes,These gochujang pork belly bites are sweet and...,"[""1 pound pork belly"", ""1/4 cup gochujang"", ""2...","[""Preheat an air fryer to 400 degrees F (200 d...",5,4
2,3-Ingredient Air Fryer Everything Bagel Chicke...,Air Fryer Recipes,Air Fryer Recipes,These 3-ingredient air fryer everything bagel ...,"[""1 \u00bc pounds chicken tenders"", ""1 tablesp...","[""Gather all ingredients. Preheat an air fryer...",3,4
3,Air Fryer Everything Bagel Chicken Cutlets,Air Fryer Recipes,Air Fryer Recipes,These air fryer everything bagel chicken cutle...,"[""4 chicken cutlets (about 1 pound total)"", ""s...","[""Preheat an air fryer to 400 degrees F (200 d...",9,9
4,Air Fryer Honey Sriracha Salmon Bites,Air Fryer Recipes,Air Fryer Recipes,These air fryer honey Sriracha salmon bites ar...,"[""1 tablespoon soy sauce"", ""1 tablespoon honey...","[""Preheat an air fryer to 400 degrees F (200 d...",5,5


In [58]:
import ast
df['ingredients'] = df['ingredients'].apply(ast.literal_eval)

def clean_ingredient_list(ingredients):
    cleaned = []
    for item in ingredients:
        item = item.lower()

        # Remove all numbers, fractions, and numeric symbols (like ½, ¼, etc.)
        item = re.sub(r'(\d+\/\d+|\d+\.\d+|\d+)', '', item)

        # Remove measurement units
        units = [
            'cup', 'cups', 'tablespoon', 'tablespoons', 'teaspoon', 'teaspoons',
            'pound', 'pounds', 'ounce', 'ounces', 'gram', 'grams', 'ml', 'liter',
            'liters', 'pinch', 'dash', 'slice', 'slices', 'can', 'package', 'clove',
            'cloves', 'stick', 'sticks', 'piece', 'pieces', 'bag', 'bags', 'oz'
        ]
        item = re.sub(r'\b(' + '|'.join(units) + r')\b', '', item)

        # Remove "of", "and", or other filler words (optional)
        item = re.sub(r'\b(of|and|with|to|for|from|into)\b', '', item)

        # Remove any punctuation and extra spaces
        item = re.sub(r'[^a-zA-Z\s]', '', item)
        item = re.sub(r'\s+', ' ', item).strip()

        if item:
            cleaned.append(item)
    return cleaned

In [59]:
import re

df['cleaned_ingredients'] = df['ingredients'].apply(clean_ingredient_list)
df[['ingredients', 'cleaned_ingredients']].head()


,ingredients,cleaned_ingredients
0,"[3/4 cup ketchup, 1/2 cup beer, 1 tablespoon W...","[ketchup, beer, worcestershire sauce, onion po..."
1,"[1 pound pork belly, 1/4 cup gochujang, 2 tabl...","[pork belly, gochujang, soy sauce, honey, grou..."
2,"[1 ¼ pounds chicken tenders, 1 tablespoon oliv...","[chicken tenders, olive oil, everything bagel ..."
3,"[4 chicken cutlets (about 1 pound total), salt...","[chicken cutlets about total, salt freshly gro..."
4,"[1 tablespoon soy sauce, 1 tablespoon honey, 1...","[soy sauce, honey, sriracha, rice vinegar, gra..."


In [60]:
df['ingredients_text'] = df['cleaned_ingredients'].apply(lambda x: ' '.join(x))
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
ingredient_vectors = vectorizer.fit_transform(df['ingredients_text'])

In [61]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_recipes(user_ingredients, top_n=5):
    user_input = ' '.join(user_ingredients).lower()
    user_vector = vectorizer.transform([user_input])
    
    similarities = cosine_similarity(user_vector, ingredient_vectors).flatten()
    
    top_indices = similarities.argsort()[-top_n:][::-1]
    top_scores = similarities[top_indices]
    
    recommendations = df.iloc[top_indices][[
        'recipe_title', 'cleaned_ingredients', 'description', 'directions'
    ]].copy()
    
    recommendations['similarity'] = top_scores.round(3)
    recommendations = recommendations[recommendations['similarity'] > 0.1]
    
    return recommendations

In [62]:
"""def recommend_with_missing(user_ingredients):
    recs = recommend_recipes(user_ingredients, top_n=3)
    
    if recs.empty:
        return "No recipes found. Try adding more ingredients!"
    
    best_match = recs.iloc[0]
    recipe_ingredients = set(best_match['cleaned_ingredients'])
    user_ingredients_set = set(user_ingredients)
    missing = recipe_ingredients - user_ingredients_set
    
    return {
        "recommended_recipe": best_match['recipe_title'],
        "similarity": best_match['similarity'],
        "missing_ingredients": list(missing)
    }"""
def recommend_with_missing(user_ingredients, top_n=3):
    recs = recommend_recipes(user_ingredients, top_n=top_n)
    
    if recs.empty:
        return "No recipes found. Try adding more ingredients!"
    
    recommendations_with_missing = []
    user_ingredients_set = set(user_ingredients)
    
    # Loop through each of the top recipes
    for _, row in recs.iterrows():
        recipe_ingredients = set(row['cleaned_ingredients'])
        missing = list(recipe_ingredients - user_ingredients_set)
        
        recommendations_with_missing.append({
            "recipe_title": row['recipe_title'],
            "similarity": row['similarity'],
            "missing_ingredients": missing,
            "description": row['description'],
            "directions": row['directions']
        })
    
    return recommendations_with_missing
    

In [63]:
user_input = ["chicken", "garlic", "soy sauce"]
results = recommend_with_missing(user_input, top_n=3)

# --- Display results ---
if isinstance(results, str):
    print(results)
else:
    for i, r in enumerate(results, 1):
        print(f"\n🔸 {i}. {r['recipe_title']} (Similarity: {r['similarity']})")
        if r['missing_ingredients']:
            print(f"   Missing ingredients: {', '.join(r['missing_ingredients'])}")
        else:
            print("   ✅ You have all ingredients!")
        print(f"📝 Description: {r['description']}")
        print(f"👩‍🍳 Directions: {r['directions'][:250]}...")  # truncated
        print("-" * 90)


🔸 1. Bat Wings (Similarity: 0.619)
   Missing ingredients: garlic powder, white sugar, chicken wings
📝 Description: I take bat wings to Halloween parties, angel wings to Christmas parties, and so on. They are just fab and easy with only 3 ingredients. My boyfriend loves to tweak this recipe. However, I just love this simple basic recipe and I hope you do, too. It's better if you can marinate this overnight. I put it in gallon sealable bag and keep in refrigerator then pour in pan the next day.
👩‍🍳 Directions: ["Preheat an oven to 375 degrees F (190 degrees C); prepare a 9x13 inch glass baking dish with cooking spray.", "Whisk together the sugar, soy sauce, and garlic powder in a medium bowl. Arrange the wings in the bottom of the prepared dish; pour the s...
------------------------------------------------------------------------------------------

🔸 2. Malaysian Barbecue Chicken Wings (Similarity: 0.593)
   Missing ingredients: dried basil, chicken wings, garlic powder, onion powder,

In [65]:
import os
import pickle
os.makedirs("C:/Users/a.nimse/Desktop/Recipe_Recommender-1/models", exist_ok=True)
pickle_path = os.path.join("C:/Users/a.nimse/Desktop/Recipe_Recommender-1/models", "recipe_recommender.pkl")


# Data to save
data_to_save = {
    "vectorizer": vectorizer,
    "ingredient_vectors": ingredient_vectors,
    "df": df
}
with open(pickle_path, "wb") as f:
    pickle.dump(data_to_save, f)

print(f"Pickle file saved successfully at: {pickle_path}")

Pickle file saved successfully at: C:/Users/a.nimse/Desktop/Recipe_Recommender-1/models\recipe_recommender.pkl
